# 🎯 Regresión Logística: Clasificación Binaria

**Módulo 03** | **Sesión 5** | **Duración estimada: 1h**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gonzalezulises/formacion-docente-bi-faces/blob/main/modulo-03-modelado-predictivo/notebooks/03_03_regresion_logistica.ipynb)

## 🎯 Objetivos de Aprendizaje

| # | Competencia | Descripción | Nivel |
|---|-------------|-------------|-------|
| 1 | Clasificación binaria | Distinguir entre problemas de regresión y clasificación | Comprensión |
| 2 | Regresión logística | Entrenar un modelo de regresión logística con scikit-learn | Aplicación |
| 3 | Interpretación | Interpretar los coeficientes y su efecto sobre la probabilidad | Análisis |
| 4 | Umbral de decisión | Comprender y ajustar el umbral de clasificación según el contexto | Evaluación |
| 5 | Caso aplicado | Construir un sistema de alerta temprana de riesgo estudiantil | Síntesis |

## 💡 ¿Por qué es importante para tu práctica docente?

Muchas de las preguntas más relevantes en el contexto universitario son de tipo **sí o no**:

- ¿Este estudiante **aprobará o reprobará**?
- ¿Esta empresa **será rentable o no**?
- ¿Este docente **está en riesgo de burnout**?
- ¿Un solicitante de crédito **pagará o no**?

La **regresión logística** es el modelo estándar para responder este tipo de preguntas. A pesar de su nombre (que incluye "regresión"), es un modelo de **clasificación**: predice a qué categoría pertenece una observación.

Su principal ventaja es que es **interpretable**: puedes explicar exactamente por qué el modelo toma cada decisión, algo crucial cuando esas decisiones afectan a personas.

In [ ]:
# Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    precision_score, recall_score, f1_score, ConfusionMatrixDisplay
)

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print('Librerías cargadas correctamente ✓')

---

## 📖 Sección 1: ¿Qué es clasificación?

En machine learning hay dos grandes tipos de problemas:

| Tipo | Pregunta | Variable objetivo | Ejemplo |
|------|----------|------------------|---------|
| **Regresión** | ¿Cuánto? | Número continuo | Predecir el promedio (12.5, 15.3...) |
| **Clasificación** | ¿Cuál? | Categoría | Predecir si aprueba o reprueba |

### Clasificación binaria

La **clasificación binaria** es el caso más simple: solo hay **dos categorías posibles**.

Ejemplos:
- Spam / No spam
- Aprobado / Reprobado
- En riesgo / Sin riesgo
- Rentable / No rentable
- Fraude / No fraude

Usualmente codificamos: la clase de interés = **1** (positivo), la otra = **0** (negativo).

---

## 🔢 Sección 2: Regresión Logística

### ¿Por qué se llama "regresión" si es clasificación?

Porque internamente, la regresión logística **calcula un número continuo** (la probabilidad) y luego lo convierte en una categoría usando un umbral.

### La función sigmoide

La regresión logística usa una función especial llamada **sigmoide** que transforma cualquier número en un valor entre 0 y 1 (una probabilidad):

$$P(y=1) = \frac{1}{1 + e^{-(b_0 + b_1 x_1 + b_2 x_2 + ...)}}$$

No necesitas memorizar la fórmula. Lo importante es entender que:

- La salida siempre está entre **0 y 1**
- Un valor cercano a 1 = alta probabilidad de pertenecer a la clase positiva
- Un valor cercano a 0 = baja probabilidad
- Por defecto, si la probabilidad es >= 0.5, clasificamos como positivo

In [ ]:
# Visualizar la función sigmoide
x_sig = np.linspace(-8, 8, 100)
y_sig = 1 / (1 + np.exp(-x_sig))

plt.figure(figsize=(10, 5))
plt.plot(x_sig, y_sig, color='steelblue', linewidth=3)
plt.axhline(y=0.5, color='red', linestyle='--', alpha=0.7, label='Umbral = 0.5')
plt.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
plt.axhline(y=1, color='gray', linestyle='-', alpha=0.3)
plt.fill_between(x_sig, y_sig, 0.5, where=(y_sig >= 0.5), alpha=0.2, color='green', label='Clase 1 (positivo)')
plt.fill_between(x_sig, y_sig, 0.5, where=(y_sig < 0.5), alpha=0.2, color='red', label='Clase 0 (negativo)')
plt.xlabel('Valor del modelo (combinación lineal)')
plt.ylabel('Probabilidad')
plt.title('La Función Sigmoide')
plt.legend()
plt.tight_layout()
plt.show()

print('La curva en S convierte cualquier valor en una probabilidad entre 0 y 1')
print('Si la probabilidad >= 0.5 → clasificamos como positivo (clase 1)')

---

## 🏛️ Sección 3: Caso — Predicción de Riesgo Estudiantil

Vamos a construir un modelo que prediga si un estudiante está **en riesgo académico** (promedio < 10).

Este tipo de modelo podría ser usado por el coordinador de carrera para identificar estudiantes que necesitan acompañamiento **antes** de que sea demasiado tarde.

In [ ]:
# Cargar datos
df = pd.read_csv('../../datasets/universidad/rendimiento_academico.csv')

# Crear variable objetivo: en_riesgo = 1 si promedio < 10
df['en_riesgo'] = (df['promedio'] < 10).astype(int)

print(f'Total de estudiantes: {len(df)}')
print(f'\nDistribución de la variable objetivo:')
print(df['en_riesgo'].value_counts().rename({0: 'Sin riesgo (0)', 1: 'En riesgo (1)'}))
print(f'\nPorcentaje en riesgo: {df["en_riesgo"].mean()*100:.1f}%')

In [ ]:
# Seleccionar features
# Convertir booleanos a números
df['beca_num'] = df['beca'].astype(int)
df['trabaja_num'] = df['trabaja'].astype(int)

features = ['semestre', 'edad', 'asistencia_pct', 'materias_inscritas', 'beca_num', 'trabaja_num']
X = df[features]
y = df['en_riesgo']

# Dividir en train y test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Datos de entrenamiento: {len(X_train)}')
print(f'Datos de prueba: {len(X_test)}')
print(f'\nProporción en riesgo (train): {y_train.mean()*100:.1f}%')
print(f'Proporción en riesgo (test):  {y_test.mean()*100:.1f}%')
print('\n(stratify=y asegura que ambos conjuntos tengan la misma proporción)')

---

## 🎓 Sección 4: Entrenar y Evaluar

Entrenemos la regresión logística y evaluemos su rendimiento.

In [ ]:
# Entrenar el modelo
modelo = LogisticRegression(max_iter=1000, random_state=42)
modelo.fit(X_train, y_train)

# Predecir en test
y_pred = modelo.predict(X_test)

# Accuracy
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy: {acc:.4f} ({acc*100:.1f}% de aciertos)')

In [ ]:
# Matriz de confusión
fig, ax = plt.subplots(figsize=(7, 6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=['Sin riesgo', 'En riesgo'],
            yticklabels=['Sin riesgo', 'En riesgo'],
            annot_kws={'size': 20}, ax=ax)
ax.set_xlabel('Predicción del modelo', fontsize=13)
ax.set_ylabel('Valor real', fontsize=13)
ax.set_title('Matriz de Confusión — Riesgo Estudiantil', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Reporte completo
print('Reporte de Clasificación')
print('=' * 55)
print(classification_report(y_test, y_pred, 
                            target_names=['Sin riesgo', 'En riesgo']))

# Interpretación en contexto
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print('Interpretación práctica:')
print(f'  Precision = {prec:.2f} → De los estudiantes señalados como "en riesgo",')
print(f'    el {prec*100:.0f}% realmente lo están (el resto son falsas alarmas)')
print(f'  Recall = {rec:.2f} → De todos los estudiantes realmente en riesgo,')
print(f'    el modelo detecta al {rec*100:.0f}% (el resto se escapan)')
print(f'  F1 = {f1:.2f} → Balance entre precision y recall')

---

## 📊 Sección 5: Coeficientes del Modelo

Una de las grandes ventajas de la regresión logística es que podemos **interpretar los coeficientes**:

- **Coeficiente positivo** → aumenta la probabilidad de estar en riesgo
- **Coeficiente negativo** → disminuye la probabilidad de estar en riesgo
- **Mayor magnitud** → mayor influencia

In [ ]:
# Tabla de coeficientes
coef_df = pd.DataFrame({
    'Variable': features,
    'Coeficiente': modelo.coef_[0],
    'Magnitud': np.abs(modelo.coef_[0])
}).sort_values('Magnitud', ascending=False)

# Agregar interpretación
coef_df['Efecto'] = coef_df['Coeficiente'].apply(
    lambda x: 'Aumenta riesgo' if x > 0 else 'Disminuye riesgo'
)

print(f'Intercepto: {modelo.intercept_[0]:.4f}\n')
print('Coeficientes ordenados por importancia:')
coef_df[['Variable', 'Coeficiente', 'Efecto']]

In [ ]:
# Visualizar coeficientes
plt.figure(figsize=(10, 5))
colores = ['#e74c3c' if c > 0 else '#2ecc71' for c in coef_df['Coeficiente']]
plt.barh(coef_df['Variable'], coef_df['Coeficiente'], color=colores)
plt.xlabel('Coeficiente')
plt.title('Factores que influyen en el riesgo estudiantil')
plt.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

print('Rojo  = aumenta la probabilidad de estar en riesgo')
print('Verde = disminuye la probabilidad de estar en riesgo')

---

## 🎚️ Sección 6: Umbral de Decisión

Por defecto, la regresión logística clasifica como positivo si la probabilidad es >= **0.5**.

Pero este umbral se puede **ajustar**:

- **Umbral más bajo** (ej: 0.3) → más sensible, detecta más casos positivos, pero más falsas alarmas
- **Umbral más alto** (ej: 0.7) → más conservador, menos falsas alarmas, pero pierde algunos casos

### ¿Cuándo bajar el umbral?

Cuando el costo de **no detectar** un caso positivo es alto. Ejemplo: si no detectamos a un estudiante en riesgo, puede reprobar o abandonar la carrera.

### ¿Cuándo subir el umbral?

Cuando los recursos son limitados y no podemos atender a todos los señalados. Solo queremos los casos más seguros.

In [ ]:
# Obtener probabilidades en lugar de predicciones binarias
y_proba = modelo.predict_proba(X_test)[:, 1]  # Probabilidad de clase 1 (en riesgo)

print('Primeras 10 probabilidades predichas:')
for i in range(10):
    estado = 'EN RIESGO' if y_test.iloc[i] == 1 else 'Sin riesgo'
    print(f'  Estudiante {i+1}: P(riesgo) = {y_proba[i]:.3f} | Real: {estado}')

In [ ]:
# Comparar diferentes umbrales
umbrales = [0.3, 0.4, 0.5, 0.6, 0.7]

resultados = []
for umbral in umbrales:
    y_pred_u = (y_proba >= umbral).astype(int)
    resultados.append({
        'Umbral': umbral,
        'Precision': precision_score(y_test, y_pred_u, zero_division=0),
        'Recall': recall_score(y_test, y_pred_u, zero_division=0),
        'F1': f1_score(y_test, y_pred_u, zero_division=0),
        'Positivos predichos': y_pred_u.sum()
    })

df_umbrales = pd.DataFrame(resultados)
print('Efecto de cambiar el umbral de decisión:')
df_umbrales

In [ ]:
# Visualizar el trade-off
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df_umbrales['Umbral'], df_umbrales['Precision'], 'o-', color='steelblue', 
        linewidth=2, label='Precision', markersize=8)
ax.plot(df_umbrales['Umbral'], df_umbrales['Recall'], 's-', color='coral', 
        linewidth=2, label='Recall', markersize=8)
ax.plot(df_umbrales['Umbral'], df_umbrales['F1'], '^-', color='green', 
        linewidth=2, label='F1 Score', markersize=8)
ax.set_xlabel('Umbral de Decisión')
ax.set_ylabel('Valor de la Métrica')
ax.set_title('Trade-off entre Precision y Recall según el Umbral')
ax.legend(fontsize=11)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

print('Umbral bajo → más recall (detecta más) pero menos precision (más falsas alarmas)')
print('Umbral alto → más precision (menos falsas alarmas) pero menos recall (pierde casos)')

---

## 💰 Sección 7: Caso Aplicado — Aprobación Crediticia

Pasemos a un contexto empresarial. Vamos a predecir si una empresa tendrá **pérdidas** (utilidad neta negativa) o **ganancias**, utilizando sus indicadores financieros.

Este tipo de análisis es común en las materias de Finanzas, Análisis de Estados Financieros y Contabilidad Gerencial de FACES.

In [ ]:
# Cargar datos financieros
fin = pd.read_csv('../../datasets/empresarial/estados_financieros.csv')
print(f'Registros: {len(fin)}')
fin.head()

In [ ]:
# Crear variable objetivo: pérdida = utilidad_neta < 0
fin['perdida'] = (fin['utilidad_neta_musd'] < 0).astype(int)

# Crear ratios financieros como features
fin['endeudamiento'] = fin['pasivos_musd'] / fin['activos_musd']
fin['margen_neto'] = fin['utilidad_neta_musd'] / fin['ingresos_musd']
fin['rotacion_activos'] = fin['ingresos_musd'] / fin['activos_musd']
fin['ratio_costos'] = fin['costos_musd'] / fin['ingresos_musd']

# Eliminar filas con infinitos o NaN
fin = fin.replace([np.inf, -np.inf], np.nan).dropna()

print(f'Distribución de la variable objetivo:')
print(fin['perdida'].value_counts().rename({0: 'Ganancia (0)', 1: 'Pérdida (1)'}))
print(f'\n{fin["perdida"].mean()*100:.1f}% de las observaciones reportan pérdidas')

In [ ]:
# Entrenar modelo de clasificación financiera
# Nota: no incluimos margen_neto porque contiene la variable objetivo
features_fin = ['endeudamiento', 'rotacion_activos', 'ratio_costos']
X_fin = fin[features_fin]
y_fin = fin['perdida']

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_fin, y_fin, test_size=0.2, random_state=42
)

modelo_fin = LogisticRegression(max_iter=1000, random_state=42)
modelo_fin.fit(X_train_f, y_train_f)
y_pred_f = modelo_fin.predict(X_test_f)

# Resultados
print('Reporte de Clasificación — Predicción de Pérdidas')
print('=' * 55)
print(classification_report(y_test_f, y_pred_f, 
                            target_names=['Ganancia', 'Pérdida']))

In [ ]:
# Coeficientes del modelo financiero
coef_fin = pd.DataFrame({
    'Variable': features_fin,
    'Coeficiente': modelo_fin.coef_[0],
    'Efecto': ['Aumenta prob. pérdida' if c > 0 else 'Disminuye prob. pérdida' 
               for c in modelo_fin.coef_[0]]
})

print('Coeficientes del modelo financiero:')
coef_fin

---

## ✏️ Ejercicios

### Ejercicio 1: Predecir ausentismo

Usando `rrhh_nomina.csv`, construye un modelo de regresión logística para predecir **alto ausentismo** (ausencias_anuales > 5).

1. Carga los datos y crea la variable objetivo binaria
2. Selecciona features relevantes (edad, antiguedad, salario_mensual_usd, evaluacion_desempeno)
3. Divide en train/test (80/20)
4. Entrena el modelo y muestra la matriz de confusión
5. Interpreta: ¿qué variables aumentan la probabilidad de alto ausentismo?

In [ ]:
# Ejercicio 1: Predecir alto ausentismo
# Tu código aquí


### Ejercicio 2: Experimentar con umbrales

Usando el modelo de riesgo estudiantil de la Sección 3:

1. Prueba umbrales de 0.2 a 0.8 (de 0.1 en 0.1)
2. Para cada umbral, calcula precision, recall y F1
3. ¿Qué umbral maximiza el F1 Score?
4. Si tu objetivo es **no dejar pasar a ningún estudiante en riesgo**, ¿qué umbral elegirías?

In [ ]:
# Ejercicio 2: Experimentar con umbrales
# Tu código aquí


---

## 📋 Resumen

| Concepto | Punto clave |
|----------|-------------|
| **Clasificación** | Predice una categoría (no un número) |
| **Regresión logística** | A pesar del nombre, es un clasificador binario |
| **Sigmoide** | Convierte cualquier valor en una probabilidad (0 a 1) |
| **Coeficientes** | Positivo → aumenta probabilidad de clase 1; Negativo → la disminuye |
| **Umbral** | Por defecto 0.5; se puede ajustar según el contexto |
| **Precision vs Recall** | Bajar umbral → más recall; Subir umbral → más precision |
| **Aplicaciones FACES** | Riesgo estudiantil, rentabilidad empresarial, ausentismo |

## 📚 Referencias

1. James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An Introduction to Statistical Learning* (2nd ed.). Springer. Capítulo 4: Classification.

2. Scikit-learn developers. (2024). *LogisticRegression*. https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html

3. Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (3rd ed.). O'Reilly. Capítulo 3: Classification.

4. Hosmer, D. W., Lemeshow, S., & Sturdivant, R. X. (2013). *Applied Logistic Regression* (3rd ed.). Wiley.